In [2]:
# Read the validated Silver Delta table

silver_df = spark.table("dbo.weather_forecast_silver")

# Inspect the schema
silver_df.printSchema()

# Preview five records
silver_df.show(5, truncate=False)

StatementMeta(, 9a80ab5b-947a-40be-ae25-a62e786d7fa7, 5, Finished, Available, Finished, False)

root
 |-- event_time: timestamp (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- humidity_pct: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- ingestion_date: date (nullable = true)

+-------------------+-------------+------------+---------+--------------+
|event_time         |temperature_c|humidity_pct|city     |ingestion_date|
+-------------------+-------------+------------+---------+--------------+
|2026-11-07 15:00:00|10.0         |81          |edinburgh|2026-11-07    |
|2026-11-07 06:00:00|9.1          |90          |edinburgh|2026-11-07    |
|2026-11-07 05:00:00|9.0          |91          |edinburgh|2026-11-07    |
|2026-11-07 14:00:00|9.9          |83          |edinburgh|2026-11-07    |
|2026-11-08 08:00:00|8.7          |93          |edinburgh|2026-11-07    |
+-------------------+-------------+------------+---------+--------------+
only showing top 5 rows


In [3]:
from pyspark.sql import functions as F

# Extract the forecast date from the hourly timestamp
daily_df = silver_df.withColumn(
    "forecast_date",
    F.to_date("event_time")
)

# Aggregate hourly forecasts into daily summaries
gold_df = (
    daily_df
    .groupBy("city", "forecast_date")
    .agg(
        F.round(F.avg("temperature_c"), 2).alias("avg_temp_c"),
        F.min("temperature_c").alias("min_temp_c"),
        F.max("temperature_c").alias("max_temp_c"),
        F.round(F.avg("humidity_pct"), 2).alias("avg_humidity_pct"),
        F.count("*").alias("hourly_readings")
    )
    .orderBy("city", "forecast_date")
)

gold_df.show(truncate=False)

StatementMeta(, 9a80ab5b-947a-40be-ae25-a62e786d7fa7, 6, Finished, Available, Finished, False)

+---------+-------------+----------+----------+----------+----------------+---------------+
|city     |forecast_date|avg_temp_c|min_temp_c|max_temp_c|avg_humidity_pct|hourly_readings|
+---------+-------------+----------+----------+----------+----------------+---------------+
|edinburgh|2026-11-07   |9.26      |8.4       |10.5      |88.08           |24             |
|edinburgh|2026-11-08   |8.66      |7.5       |9.5       |89.54           |24             |
+---------+-------------+----------+----------+----------+----------------+---------------+



In [4]:
# 1. Check the number of Gold records
gold_count = gold_df.count()

print(f"Gold record count: {gold_count}")


# 2. Check the number of hours represented by each daily aggregate
gold_df.select(
    "city",
    "forecast_date",
    "hourly_readings"
).show()


# 3. Compare source and aggregated record counts
silver_count = silver_df.count()

aggregated_count = (
    gold_df
    .agg(F.sum("hourly_readings").alias("total"))
    .first()["total"]
)

print(f"Silver records: {silver_count}")
print(f"Hourly records represented in Gold: {aggregated_count}")

assert silver_count == aggregated_count, (
    "Validation failed: Gold aggregation does not account "
    "for all Silver records"
)

print("Gold aggregation validation passed")

StatementMeta(, 9a80ab5b-947a-40be-ae25-a62e786d7fa7, 7, Finished, Available, Finished, False)

Gold record count: 2
+---------+-------------+---------------+
|     city|forecast_date|hourly_readings|
+---------+-------------+---------------+
|edinburgh|   2026-11-07|             24|
|edinburgh|   2026-11-08|             24|
+---------+-------------+---------------+

Silver records: 48
Hourly records represented in Gold: 48
Gold aggregation validation passed


In [5]:
table_name = "dbo.weather_daily_gold"

if not spark.catalog.tableExists(table_name):

    # First execution: create and populate the Gold table
    (
        gold_df.write
        .format("delta")
        .mode("errorifexists")
        .saveAsTable(table_name)
    )

    print("Gold table created successfully")

else:

    # Subsequent executions: refresh existing Gold records
    gold_df.createOrReplaceTempView("gold_updates")

    spark.sql("""
        MERGE INTO dbo.weather_daily_gold AS t
        USING gold_updates AS s
        ON t.city = s.city
           AND t.forecast_date = s.forecast_date

        WHEN MATCHED THEN UPDATE SET *

        WHEN NOT MATCHED THEN INSERT *
    """)

    print("Gold table refreshed successfully")

StatementMeta(, 9a80ab5b-947a-40be-ae25-a62e786d7fa7, 8, Finished, Available, Finished, False)

Gold table refreshed successfully


In [6]:
saved_gold_df = spark.table("dbo.weather_daily_gold")

saved_gold_df.orderBy("forecast_date").show(
    truncate=False
)

StatementMeta(, 9a80ab5b-947a-40be-ae25-a62e786d7fa7, 9, Finished, Available, Finished, False)

+---------+-------------+----------+----------+----------+----------------+---------------+
|city     |forecast_date|avg_temp_c|min_temp_c|max_temp_c|avg_humidity_pct|hourly_readings|
+---------+-------------+----------+----------+----------+----------------+---------------+
|edinburgh|2026-11-07   |9.26      |8.4       |10.5      |88.08           |24             |
|edinburgh|2026-11-08   |8.66      |7.5       |9.5       |89.54           |24             |
+---------+-------------+----------+----------+----------+----------------+---------------+



In [7]:
# Read the persisted Gold table
saved_gold_df = spark.table("dbo.weather_daily_gold")

print("Total Gold records:", saved_gold_df.count())

# Check duplicate business keys
duplicates_df = (
    saved_gold_df
    .groupBy("city", "forecast_date")
    .count()
    .filter(F.col("count") > 1)
)

duplicates_df.show()

StatementMeta(, 9a80ab5b-947a-40be-ae25-a62e786d7fa7, 10, Finished, Available, Finished, False)

Total Gold records: 2
+----+-------------+-----+
|city|forecast_date|count|
+----+-------------+-----+
+----+-------------+-----+

